# Notebook 3: Model Evaluation & Comparison

Computes mAP, IoU, Precision-Recall curves, and error analysis for all three models.
Generates the final comparison table required for the report.


In [1]:
import sys, time
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torchvision.transforms.functional as TF
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm

PROJECT_ROOT = Path('..').resolve()
DATA_DIR     = PROJECT_ROOT / 'data'
RUNS_DIR     = PROJECT_ROOT / 'runs'
sys.path.insert(0, str(PROJECT_ROOT))

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES    = ['cracks', 'spalling', 'corrosion', 'potholes', 'paint_degradation']
NUM_CLASSES = len(CLASSES) + 1
CONF_THRESH = 0.25
IOU_THRESH  = 0.50
IMG_SIZE    = 640

## 1. Load Models


In [2]:
import csv as _csv
from ultralytics import YOLO
from models.ssdlite_detector import build_model as build_ssd


def _best_yolo_weights(model_dir: Path) -> Path:
    """Return best.pt from the vN with highest mAP@0.5; falls back to latest version."""
    versions = sorted(
        [d for d in model_dir.iterdir() if d.is_dir() and d.name.startswith('v')],
        key=lambda d: int(d.name[1:])
    )
    if not versions:
        raise FileNotFoundError(f'No runs in {model_dir}')
    best_ver, best_map = versions[-1], -1.0
    for v in versions:
        results = v / 'results.csv'
        if not results.exists() or results.stat().st_size == 0:
            continue
        try:
            with open(results) as f:
                rows = list(_csv.DictReader(f))
            v_best = max(float(r.get('metrics/mAP50(B)', 0)) for r in rows) if rows else 0.0
            if v_best > best_map:
                best_map, best_ver = v_best, v
        except Exception:
            continue
    return best_ver / 'weights' / 'best.pt'


def _latest_ssd_weights(model_dir: Path) -> Path:
    ckpts = sorted(model_dir.glob('best_v*.pth'), key=lambda p: int(p.stem.split('_v')[1]))
    if not ckpts:
        raise FileNotFoundError(f'No checkpoints in {model_dir}')
    return ckpts[-1]


models = {}

for model_name, model_dir in [('YOLOv11n', RUNS_DIR / 'yolo11n'), ('YOLOv8n', RUNS_DIR / 'yolov8n')]:
    if model_dir.exists():
        try:
            w = _best_yolo_weights(model_dir)
            models[model_name] = YOLO(str(w))
            print(f'Loaded {model_name} from {w}')
        except FileNotFoundError as e:
            print(f'{model_name}: {e}')

ssd_dir = RUNS_DIR / 'ssdlite'
if ssd_dir.exists():
    try:
        w = _latest_ssd_weights(ssd_dir)
        ssd = build_ssd(num_classes=NUM_CLASSES, pretrained=False).to(DEVICE)
        ckpt = torch.load(str(w), map_location=DEVICE)
        ssd.load_state_dict(ckpt['model_state_dict'])
        ssd.eval()
        models['SSDLite'] = ssd
        print(f'Loaded SSDLite from {w}')
    except FileNotFoundError as e:
        print(f'SSDLite: {e}')

print(f'\nModels loaded: {list(models.keys())}')

Loaded YOLOv11n from C:\dev\ai_for_engineering\runs\yolo11n\v1\weights\best.pt
Loaded YOLOv8n from C:\dev\ai_for_engineering\runs\yolov8n\v1\weights\best.pt


Loaded SSDLite from C:\dev\ai_for_engineering\runs\ssdlite\best_v1.pth

Models loaded: ['YOLOv11n', 'YOLOv8n', 'SSDLite']


## 2. Evaluation Functions


In [3]:
from torch.utils.data import DataLoader, Dataset

IMAGE_EXTS = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']


class TestDataset(Dataset):
    def __init__(self, split: str = 'test'):
        self.img_dir = DATA_DIR / 'images' / split
        self.lbl_dir = DATA_DIR / 'labels' / split
        seen, imgs = set(), []
        for ext in IMAGE_EXTS:
            for p in self.img_dir.glob(ext):
                if p.name.lower() not in seen:
                    seen.add(p.name.lower())
                    imgs.append(p)
        self.images = sorted(imgs)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        lbl_path = self.lbl_dir / (img_path.stem + '.txt')
        img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        tensor = TF.to_tensor(img)
        boxes, labels = [], []
        if lbl_path.exists():
            for line in lbl_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
                x1 = max(0.0, (cx - bw / 2) * IMG_SIZE)
                y1 = max(0.0, (cy - bh / 2) * IMG_SIZE)
                x2 = min(float(IMG_SIZE), (cx + bw / 2) * IMG_SIZE)
                y2 = min(float(IMG_SIZE), (cy + bh / 2) * IMG_SIZE)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(cls + 1)
        if boxes:
            boxes_t  = torch.tensor(boxes,  dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.int64)
        else:
            boxes_t  = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros(0,      dtype=torch.int64)
        return tensor, {'boxes': boxes_t, 'labels': labels_t, 'image_id': torch.tensor([idx])}


def collate_fn(batch):
    return tuple(zip(*batch))


test_ds     = TestDataset('test')
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)
print(f'Test images: {len(test_ds)}')

Test images: 695


In [4]:
@torch.no_grad()
def evaluate_torchvision(model, loader, device, conf=CONF_THRESH):
    model.eval()
    metric = MeanAveragePrecision(iou_type='bbox', class_metrics=True)
    times = []
    for images, targets in tqdm(loader):
        imgs = [img.to(device) for img in images]
        t0 = time.time()
        outputs = model(imgs)
        times.append((time.time() - t0) / len(imgs) * 1000)  # ms/image
        preds = []
        for o in outputs:
            mask = o['scores'] >= conf
            preds.append({'boxes':  o['boxes'][mask].cpu(),
                          'scores': o['scores'][mask].cpu(),
                          'labels': o['labels'][mask].cpu()})
        gts = [{'boxes':  t['boxes'].cpu(),
                'labels': t['labels'].cpu()} for t in targets]
        metric.update(preds, gts)
    result = metric.compute()
    result['inference_ms'] = np.mean(times)
    return result


def evaluate_yolo(model, data_yaml, split='test'):
    return model.val(data=data_yaml, split=split)


def measure_yolo_inference_speed(model, n_images: int = 50):
    imgs = list((DATA_DIR / 'images' / 'test').glob('*.jpg'))[:n_images]
    times = []
    for p in imgs:
        t0 = time.time()
        model.predict(str(p), verbose=False)
        times.append((time.time() - t0) * 1000)
    return np.mean(times)

## 3. Run Evaluation on Test Set


In [5]:
DATA_YAML = str(DATA_DIR / 'data.yaml')
results = {}

for model_name, model in models.items():
    if model_name in ('YOLOv11n', 'YOLOv8n'):
        print(f'\n--- Evaluating {model_name} ---')
        val   = model.val(data=DATA_YAML, split='test')
        speed = measure_yolo_inference_speed(model)
        results[model_name] = {
            'mAP@0.5':      round(float(val.box.map50), 4),
            'mAP@0.5:0.95': round(float(val.box.map),   4),
            'Precision':    round(float(val.box.mp),     4),
            'Recall':       round(float(val.box.mr),     4),
            'Speed (ms)':   round(speed, 2),
        }
    elif model_name == 'SSDLite':
        print(f'\n--- Evaluating {model_name} ---')
        r = evaluate_torchvision(model, test_loader, DEVICE)
        results[model_name] = {
            'mAP@0.5':      round(float(r['map_50']), 4),
            'mAP@0.5:0.95': round(float(r['map']),    4),
            'Precision':    None,
            'Recall':       None,
            'Speed (ms)':   round(float(r['inference_ms']), 2),
        }


--- Evaluating YOLOv11n ---
Ultralytics 8.4.38  Python-3.11.2 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)


YOLO11n summary (fused): 101 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs


val: Fast image access  (ping: 0.30.0 ms, read: 82.626.1 MB/s, size: 66.9 KB)


val: Scanning C:\dev\ai_for_engineering\data\labels\test... 49 images, 0 backgrounds, 0 corrupt: 7% ╸─────────── 49/695 146.1it/s 0.1s<4.4s

val: Scanning C:\dev\ai_for_engineering\data\labels\test... 136 images, 0 backgrounds, 0 corrupt: 20% ━━────────── 136/695 362.8it/s 0.2s<1.5s

...

 98%|█████████▊| 171/174 [00:27<00:00,  5.91it/s]

 99%|█████████▉| 172/174 [00:27<00:00,  5.66it/s]

 99%|█████████▉| 173/174 [00:27<00:00,  5.59it/s]

100%|██████████| 174/174 [00:27<00:00,  5.74it/s]

100%|██████████| 174/174 [00:27<00:00,  6.28it/s]

## 4. Model Comparison Table


In [6]:
comparison_df = pd.DataFrame(results).T
comparison_df.index.name = 'Model'
print('\n=== Model Comparison Table ===')
print(comparison_df.to_string())

comparison_df.to_csv(RUNS_DIR / 'model_comparison.csv')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
metrics_to_plot = ['mAP@0.5', 'mAP@0.5:0.95']
colors = ['#3498db', '#e74c3c', '#2ecc71']
for i, metric in enumerate(metrics_to_plot):
    vals = [results[m][metric] for m in results if results[m][metric] is not None]
    names = [m for m in results if results[m][metric] is not None]
    axes[i].bar(names, vals, color=colors[:len(names)], edgecolor='black')
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].set_ylabel('Score')
    for j, v in enumerate(vals):
        axes[i].text(j, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)
plt.suptitle('Model Comparison', fontsize=14)
plt.tight_layout()
plt.savefig(str(RUNS_DIR / 'model_comparison.png'), dpi=150)
plt.show()


=== Model Comparison Table ===
          mAP@0.5  mAP@0.5:0.95  Precision  Recall  Speed (ms)
Model                                                         
YOLOv11n   0.7557        0.6719     0.7507  0.7041       42.66
YOLOv8n    0.7477        0.6592     0.7792  0.6912       31.84
SSDLite    0.0530        0.0434        NaN     NaN       20.38


<Figure size 1200x400 with 2 Axes>

## 5. Per-Class AP (YOLO)


In [7]:
colors_bar = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

for model_name in ('YOLOv11n', 'YOLOv8n'):
    if model_name not in models:
        continue
    val_res = models[model_name].val(data=DATA_YAML, split='test')
    per_class = dict(zip(CLASSES, val_res.box.maps))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(list(per_class.keys()), list(per_class.values()), color=colors_bar, edgecolor='black')
    ax.set_title(f'{model_name} — Per-Class AP@0.5')
    ax.set_ylabel('AP@0.5')
    ax.set_ylim(0, 1)
    for i, (k, v) in enumerate(per_class.items()):
        ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
    plt.tight_layout()
    fname = f'per_class_ap_{model_name.lower()}.png'
    plt.savefig(str(RUNS_DIR / fname), dpi=150)
    plt.show()
    print(f'Saved {fname}')

Ultralytics 8.4.38  Python-3.11.2 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)


val: Fast image access  (ping: 0.10.0 ms, read: 210.0161.1 MB/s, size: 43.2 KB)


val: Scanning C:\dev\ai_for_engineering\data\labels\test.cache... 695 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 695/695  0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 2% ──────────── 1/44 1.7s/it 0.5s<1:12

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 5% ╸─────────── 2/44 1.3it/s 0.9s<32.9s

...

     paint_degradation         58         58      0.843          1      0.971      0.971


Speed: 2.2ms preprocess, 5.4ms inference, 0.0ms loss, 2.1ms postprocess per image


Results saved to C:\dev\ai_for_engineering\notebooks\runs\detect\val4


<Figure size 800x400 with 1 Axes>

Saved per_class_ap_yolov8n.png


## 6. Severity Distribution on Test Set


In [8]:
def severity_label(bbox_w_norm: float, bbox_h_norm: float) -> str:
    """Returns Low/Medium/High based on fractional bbox area."""
    area_pct = bbox_w_norm * bbox_h_norm * 100
    if area_pct < 5:
        return 'Low'
    elif area_pct <= 20:
        return 'Medium'
    return 'High'


severity_counts = {'Low': 0, 'Medium': 0, 'High': 0}
for lbl_path in (DATA_DIR / 'labels' / 'test').glob('*.txt'):
    for line in lbl_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) == 5:
            bw, bh = float(parts[3]), float(parts[4])
            severity_counts[severity_label(bw, bh)] += 1

print('Ground-truth severity distribution in test set:')
for sev, cnt in severity_counts.items():
    print(f'  {sev}: {cnt}')

fig, ax = plt.subplots(figsize=(5, 4))
ax.pie(severity_counts.values(), labels=severity_counts.keys(),
       autopct='%1.1f%%', colors=['#2ecc71', '#f39c12', '#e74c3c'])
ax.set_title('Ground-Truth Severity Distribution (Test Set)')
plt.tight_layout()
plt.savefig(str(RUNS_DIR / 'severity_distribution.png'), dpi=150)
plt.show()

Ground-truth severity distribution in test set:
  Low: 587
  Medium: 160
  High: 375


<Figure size 500x400 with 1 Axes>

## 7. Qualitative Prediction Visualisation (YOLO)


In [9]:
if 'YOLOv11n' in models:
    CLASS_COLORS = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']
    test_imgs = sorted((DATA_DIR / 'images' / 'test').glob('*.jpg'))[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    for i, img_path in enumerate(test_imgs):
        result = models['YOLOv11n'].predict(str(img_path), conf=CONF_THRESH, verbose=False)[0]
        img = Image.open(img_path).convert('RGB')
        axes[i].imshow(img)
        w, h = img.size
        for box in result.boxes:
            cls   = int(box.cls.item())
            conf  = float(box.conf.item())
            x1,y1,x2,y2 = box.xyxy[0].tolist()
            bw_n = (x2 - x1) / w
            bh_n = (y2 - y1) / h
            sev  = severity_label(bw_n, bh_n)
            label = f'{CLASSES[cls]} {conf:.2f} [{sev}]'
            rect  = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                       linewidth=2,
                                       edgecolor=CLASS_COLORS[cls],
                                       facecolor='none')
            axes[i].add_patch(rect)
            axes[i].text(x1, y1-5, label, color=CLASS_COLORS[cls],
                         fontsize=7, fontweight='bold')
        axes[i].axis('off')
        axes[i].set_title(img_path.name, fontsize=8)

    plt.suptitle('YOLOv11n Predictions with Severity', fontsize=13)
    plt.tight_layout()
    plt.savefig(str(RUNS_DIR / 'prediction_samples.png'), dpi=120)
    plt.show()

<Figure size 1500x800 with 6 Axes>